# Train the GPT

## 1. Get the dataset

In [ ]:
# GPT is a probabilitsic model therefore the response it generate for a prompt is non-deterministic.
with open('dataset/tiny.txt', 'r', encoding='utf-8') as f:
    text = f.read()
# Tiny Shakespeare dataset is used because it is short enought to train on the local computer but long enough i.e. 1 Million characters that it will be hard for a human to cheat/augment the model by providing the answers or regex.

### 1. See the dataset

In [29]:
print('First 500 characters of the dataset:')
print(text[:500])

First 500 characters of the dataset:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## 2. Character level Encoding and Decoding Strategy

In [30]:
# Find out how many unique characters are there in this dataset?
# For english text we expect it around 65ish characters (lowercase, uppercase, digits, punctuation, whitespace)
# For code we expect more due to brackets, oprators, special symbold etc.
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('Unique characters in the dataset are: ', ''.join(chars))
print('Unique characters in the dataset:', vocab_size)

Unique characters in the dataset are:  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Unique characters in the dataset: 65


In [67]:
# Strategy to tokenise the input text
# Assing a integer to the word that is in the dataset. This assignment is based on the vocabulary of possible elements. 
# Because we are building a character level language model, we will asign an integer to each unique character in the dataset. So if A is 1 and then D is 4, word ADD becomes 144


stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

encoding = lambda s: [stoi[c] for c in s] # Character -> Integer
decoding = lambda l: ''.join(itos[i] for i in l) # Integer -> Character, this is only a checking step

print(encoding("ADD"))
print(decoding(encoding("ADD")))

## Character level encoding is faily simple to grashp and helps to understand what is under the hood working of the model.
## One popular encoding starategy is SentencePiece which Google uses. Sentece Piece is a sub-word tokenizer which is between encoding each character and encoding entire word.
## Another popular encoding strategy is TikToken which is used by OpenAI. This uses BytePairEncoding tokeniser (BPE)
## In tikToken, instead of 65 tokens it has 50,257 tokens. This encoding strategy was used for GPT 2.
## Why this matter, you can have very long dictionary of words with very small secquence of integers. Or you have a very small dictionary wiht a large sequeence of integers.

[13, 16, 16]
ADD


## 3. Tokenise the dataset based on Encoding strategy defined in previous step

In [68]:
import torch
data = torch.tensor(encoding(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115393]) torch.int64


## 3.1 Data Tensor

In [69]:
print('CP 1: Data Tensor first 500 characters', data[:500])

CP 1: Data Tensor first 500 characters tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
 

## 3.2 Train and Test splits

In [70]:
# It is a good practise to split the dataset into train and test set. This makes sure that the mode is not overfitteing on the dataset and can generalise well. 
# Can the model generalise well and predict for the unknown data is tested by the accuracy of the model. Usually measure by accuracy score.
# There is also sometimes the concept of validation set which is used to tune the hyperparameters of the model in later stages.

n = int(0.9*(len(data)))
train_data = data[:n]
test_data = data[n:]

# In a production environment: The exact split of train and test data is a subject of experiment in iteself. 
# Depending upon the model and the dataset the seplit could be 70:30, 80:20, 90:10 and one must perform experiment will different ration of train and test data to find out the optimal split.
# For this project: 90:10 is taken because the focus is to understand the transformer architecture and the workings of it.

## 4. Load the data for training

In [71]:
# Data loading happends in chunks, this chunks have a max lenghth.
# In a production system: The exact lenght of the chunk is a hyperparamere that means it needs to be tuned by conducting experiements and find out what is the right batch size the leads to least amount of training time vs least amount of loss in training.
# For this project: 256 is taken as the batch size because it is small enough to train on the local computer and large enough to get a good accuracy score.
context_length = 8
train_data[:context_length+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [72]:
# What we want to do is, train the next character given the previous characters(s)
# example given we have 18, we want to say 47 likely comes next. 
# for 18 & 47 together 56 likely comes next etc.

x = train_data[:context_length]
y = train_data[1:context_length+1]

for t in range(context_length):
    context_length = x[:t+1]
    target = y[t]
    print(f'When the input is {context_length} the target is {target}')

# The idea to train the model from 1 to context_length is to make sure the model get use to seeing the different lenght of inputs for infeering the next character.

When the input is tensor([18]) the target is 47
When the input is tensor([18, 47]) the target is 56
When the input is tensor([18, 47, 56]) the target is 57
When the input is tensor([18, 47, 56, 57]) the target is 58
When the input is tensor([18, 47, 56, 57, 58]) the target is 1
When the input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
When the input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
When the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [73]:
# block size is the number of context that will be sent to the model for training.
# Batch is used so that we can make use of the parallel processing power of the GPUs.
# In production system: The bactch size is also a hyperprameter that needs to be tunes by conduting experiements. We are optimising for the efficiency.
torch.manual_seed(1337)
batch_size = 4
context_length = 8

# This function will be used to get the batch for the training and the test set.
def get_batch(split):
    data = train_data if split == 'train' else test_data
    offset = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length] for i in offset])
    y = torch.stack([data[i+1:i+context_length+1] for i in offset])
    return x, y

xb,yb = get_batch('train')
print('Input batch (x):')
print(xb.shape)
print(xb)
print('Target batch (y):')
print(yb.shape)
print(yb)

# The learning from out output is that there is a 4x8 array/tensor that is being used to train the model.

print('' )
for b in range(batch_size):
    for t in range(context_length):
        context = xb[b, :t+1].tolist()
        target = yb[b, t].item()
        print(f'When the input is {context} the target is {target}')

Input batch (x):
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
Target batch (y):
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])

When the input is [53] the target is 59
When the input is [53, 59] the target is 6
When the input is [53, 59, 6] the target is 1
When the input is [53, 59, 6, 1] the target is 58
When the input is [53, 59, 6, 1, 58] the target is 56
When the input is [53, 59, 6, 1, 58, 56] the target is 47
When the input is [53, 59, 6, 1, 58, 56, 47] the target is 40
When the input is [53, 59, 6, 1, 58, 56, 47, 40] the target is 59
When the input is [49] the target is 43
When the input is [49, 43] the target is 43
When the input is [49, 43, 43] the target is 54
When the input is [49, 43, 43, 54] th

In [74]:
print(xb)

tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])


## Training hardware is
MacBook Pro M1 - 2021

8-core CPU with 4 performance cores and 4 efficiency cores

8-core GPU

16-core Neural Engine

## 5. Bygram Language Model

In [93]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B,T,C); Batch, Time, Channel
        
        if targets is None:
            loss = None
        else:
            #reshapte the logits
            B,T,C = logits.shape
            logits = logits.view(B*T, C)

            #reshape the target
            targets = targets.view(B*T)
            # quality of a prediction
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    def generate(self,idx, max_new_tokens):
    # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            
            # get the predictions
            logits, loss = self(idx)
            
            # focus only on the last time step
            logits = logits[:,-1, :] # becomes (B,C)
            
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
# Expectation is -ln(1/65) = 4,17

torch.Size([32, 65])
tensor(4.8948, grad_fn=<NllLossBackward>)


In [94]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
# Expectation is -ln(1/65) = 4,17

idx = torch.zeros((1,1), dtype=torch.long)
print(decoding(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.7525, grad_fn=<NllLossBackward>)

v?vqKGh
txYXcDRSW&jxn;P Kuvbv
Bl:KuMWi;JYlFahlK3u.M,QV;oky3wcxoYPQpbpb
r3Nc 'MQHQ uOfxoxwXIvSE-$&a$K
